In [1]:
import pandas as pd
import numpy as np
import collections
import time
from scipy.spatial.distance import euclidean
import heapq
import re

import sys
sys.path.insert(0, '../../../../PwC Laptop/ESTM - Copy (3)/ESTM - Copy (2)')
import dataImporter

from dataImporter import *
from experiment import *
from utilityFunctions import *
from tsFeatures import *
from afterAnalysis import *
from visualization import *

In [2]:
def calculate_distance_matrix(df, distance_function=euclidean):

    """
    Calculate pairwise distances between all pairs of stocks in the DataFrame df
    based on their time series using the polygonal distance measure.

    Returns a NumPy array where the indices correspond to stock pairs, and the values
    are the corresponding distances.
    """

    df = df.reset_index(drop=True)
    stocks = df.index.tolist()
    num_stocks = len(stocks)
    distance_matrix = np.zeros((num_stocks, num_stocks))

    #with tqdm(total=num_stocks * (num_stocks - 1) // 2) as pbar:
    for i in range(num_stocks):
        for j in range(i + 1, num_stocks):
            ts1, ts2 = df.loc[stocks[i], 'target'], df.loc[stocks[j], 'target']
            dist = distance_function(np.array(ts1), np.array(ts2))
            distance_matrix[i, j] = dist
            distance_matrix[j, i] = dist
                #pbar.update(1)

    return distance_matrix

def calculate_intra_subgroup_distance(subgroup_df, distance_matrix):
    """
    Calculate the average distance between all pairs of stocks in the given subgroup
    based on the pre-computed distance matrix.

    Returns a float representing the average distance.
    """
    subgroup_indices = subgroup_df.index.to_numpy()
    pairwise_distances = distance_matrix[np.ix_(subgroup_indices, subgroup_indices)]
    num_pairs = len(subgroup_indices) * (len(subgroup_indices) - 1) // 2
    total_distance = np.nansum(pairwise_distances) / 2
    intra_subgroup_distance = total_distance / max(1, num_pairs)
    return intra_subgroup_distance

# @jit(target_backend='cuda')
def calculate_inter_subgroup_distance(subgroup_df, complement_df, distance_matrix):
    """
    Calculate the average distance between all pairs of stocks in the given subgroup and
    complement subgroup based on the pre-computed distance matrix.

    Returns a float representing the average distance.
    """
    subgroup_indices = subgroup_df.index.to_numpy()
    complement_indices = complement_df.index.to_numpy()
    pairwise_distances = distance_matrix[np.ix_(subgroup_indices, complement_indices)]
    num_pairs = len(subgroup_indices) * len(complement_indices)
    total_distance = np.nansum(pairwise_distances)
    inter_subgroup_distance = total_distance / max(1, num_pairs)
    return inter_subgroup_distance


def cluster_based_quality_measure(subgroup_df, col, complement_df, **kwargs):

    """calculates ratio between inter subgroup distance and intra subgroup distance,
    corrected for the size

    Based on Verhaegh 2022

    """

    n = len(subgroup_df)
    n_c = len(complement_df)
    N = n + n_c

    # if 'results' in kwargs:
    #     resultSet = kwargs['results']
    #     resultSet.values

    distance_matrix = kwargs['distance_matrix']
    size_correction = kwargs['correct_for_size']


    return size_correction(n, N) * calculate_inter_subgroup_distance(subgroup_df, complement_df, distance_matrix) \
        / (calculate_intra_subgroup_distance(subgroup_df, distance_matrix) + 1 )


In [3]:
import heapq
import re

class botumUpSearch:
    
    def __init__(self,df,distance_matrix,number_of_row_pairs,depth,q,nr_chunks=5,min_coverage_perc=0.01,min_coverage_abs=3):
    
        self.matrix = distance_matrix
        self.x = number_of_row_pairs
        self.d = depth
        self.df = df.copy()
        self.df_org = df.copy()
        self.q = q
        self.min_coverage_perc = min_coverage_perc
        self.min_coverage_abs = min_coverage_abs
        self.nr_chunks = nr_chunks
        
        # steps to partition numerical columns into bins
        for col in self.df.columns:
            if (self.df[col].dtype == 'float64') or (self.df[col].dtype == 'float32') or (self.df[col].dtype == 'int64'):
                dat = np.sort(self.df[col])
                dat = dat[np.logical_not(np.isnan(dat))]
                for i in range(1, self.nr_chunks+1):
                    # determine the number of chunks you want to divide your data in
                    x = np.percentile(dat, (i-1)*100/self.nr_chunks)  #
                    y = np.percentile(dat, (i)*100/self.nr_chunks)
                    candidate = "{} <= {} <= {}".format(x ,col, y)
                    self.df[col] = self.df[col].apply(lambda val: candidate if (not isinstance(val, str) and x <= val <= y) else val)
            
            
    def find_min_indices(matrix, x):
        """
        Finds the x smallest distances and returns tuples of the indexes i and j.
        
        matrix = distance matrix
        x = number of pairs we want to find
        """
        
        min_heap = []
        n = len(matrix)

        for i in range(n):
            for j in range(i+1, n):
                if len(min_heap) < x:
                    heapq.heappush(min_heap, (-matrix[i][j], (i, j)))
                else:
                    if matrix[i][j] < -min_heap[0][0]:
                        heapq.heappop(min_heap)
                        heapq.heappush(min_heap, (-matrix[i][j], (i, j)))

        return [idx for val, idx in min_heap]


    def get_common_attributes(rows, d_len):
        """
        Finds the attribute value combinations that are similar in all rows.

        rows = list of rows of a dataframe
        d_len = description length
        """

        common_attributes = []
        attribute_combinations = combinations(rows[0].items(), d_len)

        for combination in attribute_combinations:
            attributes = [f"{attribute} == '{value}'" for attribute, value in combination]
            if all(all(row.get(attribute) == value for attribute, value in combination) for row in rows[1:]):
                common_attributes.append(attributes)
                
        # print("common_attributes = ", common_attributes)

        return common_attributes


    
    def get_unique_lists(list_of_lists):
        """
        Filters redundant descriptions, when they are exactly the same or in a different order.
        """
        
        unique_lists = [list(x) for x in set(tuple(set(sublist)) for sublist in list_of_lists)]
        return unique_lists
    
    def generate_new_tuples(tuples_list, matrix):
        z = len(tuples_list[0])  # Size of the tuples in the original list
        new_tuples_list = []

        for t in tuples_list:
            new_tuples = []

            for index in t:
                row = index
                new_tuple = list(t)  # Create a new tuple based on the existing one

                # Find the minimum value in the row that satisfies the conditions
                min_val = float('inf')
                min_index = None
                for i, val in enumerate(matrix[row]):
                    if val < min_val and i != row and i not in t:
                        min_val = val
                        min_index = i

                new_tuple.append(min_index)  # Add the new index to the tuple
                new_tuples.append(tuple(new_tuple))  # Convert the list back to a tuple and add it to the new list

            new_tuples_list.extend(new_tuples)

        return new_tuples_list   

    
    def findQuality(self, quality_measure = cluster_based_quality_measure, z=3,comparison_type = "complement" ,size_corr = no_size_corr):
        
        self.running_time = time.time()
        self.quality_measure = quality_measure
        self.size_corr = size_corr
        self.quality_measure_counter = 0
        self.comparison_type = comparison_type
        self.z = z
        z=z-2
        
        
        min_heap = []
        len_df = len(self.df)
        
        promising_combinations = botumUpSearch.find_min_indices(self.matrix, self.x)
        print('promising_combinations_start step= ',len(promising_combinations))
        
        promising_combinations_larger_size = promising_combinations
        if z>0:
            for i in range(z):
                promising_combinations_larger_size = botumUpSearch.generate_new_tuples(promising_combinations_larger_size, self.matrix)
             
        print('promising_combinations_mid step= ',len(promising_combinations_larger_size))   
        promising_combinations_larger_size_unique = [tuple(s) for s in {frozenset(t) for t in promising_combinations_larger_size}]
        promising_combinations = promising_combinations_larger_size_unique
        
        print('promising_combinations_large= ',len(promising_combinations))
        
        

        #TODO in line below it only works when target attribute is the last attribute
        
        candidate_descriptions = [botumUpSearch.get_common_attributes([self.df.iloc[i][:-1] for i in index_tuple],d) for index_tuple in promising_combinations for d in range(1,self.d+1)]
        print('descriptions',len(candidate_descriptions))

        # unique_candidate_descriptions gives the unique potential candidate descriptions
        unique_candidate_descriptions = botumUpSearch.get_unique_lists([item for sublist in candidate_descriptions for item in sublist])
        
        print('unique descriptions',len(unique_candidate_descriptions))

        for desc in unique_candidate_descriptions:
            ind = self.df.query(as_string(desc))
            
            #checks if subgroups comply with size constrains
            if satisfies_all(desc, ind, len_df, self.min_coverage_perc, self.min_coverage_abs):
                
                quality, coverage = eval_quality(ind, self.df, 'target', self.quality_measure, comparison_type,distance_matrix=self.matrix,correct_for_size=size_corr)

                self.quality_measure_counter += 1
                
                if len(min_heap) < self.q:
                    heapq.heappush(min_heap, (quality, desc, coverage))
                    #heapq.heapify(min_heap)
                else:
                    
                    if -quality < -heapq.nsmallest(1, min_heap)[0][0]:

                        equal_quals = [i for i, x in enumerate(min_heap) if x[0] == quality]
                        
                        # checks if set of records isn't already present as a result of a different description
                        if len(equal_quals)>0:
                            for i in equal_quals:
                                comp = self.df.query(as_string(min_heap[i][1]))
                                if np.array_equal(comp.index, ind.index):
                                    pass
                                else:
                                    heapq.heappushpop(min_heap, (quality, desc, coverage))
                        else:
                            heapq.heappushpop(min_heap, (quality, desc, coverage))

        
        self.running_time = time.time() - self.running_time
        
        self.result = sorted(min_heap, key=lambda x: x[0], reverse=True)
        
        
        #part that changes the numerical propositions back into a evaluatable thing instead of a string
        data = bus.result
        
        for i, (value1, sublist, value2) in enumerate(data):
            for j, string in enumerate(sublist):
                
                if any(op in string for op in ['<', '<=', '>', '>=']):
                    # Extract the nested string without the outer quotes
                    try:
                        nested_string = string.split("'")[1]
                     # Replace the string with the extracted nested string
                        sublist[j] = nested_string
                    except:
                        pass
             # Update the modified sublist in the data
            data[i] = ( value1, sublist,value2)
            
        
        self.result = data
        self.quals = [i[0] for i in min_heap]
        self.covs = [i[2] for i in min_heap]
        self.avg_quality = sum(self.quals)/len(self.quals)
        self.avg_coverage = sum(self.covs)/len(self.covs)
        self.descs = [i[1] for i in min_heap]
        
    def print_outcome(self):
        
        print('after checking ',self.quality_measure_counter,' potential subgroups, in ',round(self.running_time,3),' seconds')
        print('Outcome of bottumUpSearch is:')
        print(' ')
        print('avg_quality = ',round(self.avg_quality,3))
        print('max_quality = ',round(max(self.quals),3))
        print(' ')
        print('avg_coverage = ',round(self.avg_coverage,3))
        print('max_coverage = ',round(max(self.covs),3))
        print(' ')
                
        for z in self.result:
            conjunction = " Ʌ ".join(["(" + condition.replace(" == ", "=").strip() + ")" for condition in z[1]])
            print('description =',conjunction)
            print('quality =',round(z[0],3))
            print('coverage =',round(z[2],3))
            print(' ')
            
    def statistical_test(self,k,save_title="./temp_statistical_test_results.pkl"):
        
        """
        Creates quality measure outputs of k randomized experiments 
        to use for statistical testing of the original output.
        """
        
        bus_df = self.df.reset_index(drop=True)

        results = []

        for _ in range(k):
            
            matrix = botumUpSearch.randomize_symmetric_matrix(self.matrix)

            bus_rand = botumUpSearch(bus_df, matrix, self.x, self.d, 1,
                                     nr_chunks=self.nr_chunks, min_coverage_perc=self.min_coverage_perc,
                                     min_coverage_abs=self.min_coverage_abs)
            bus_rand.findQuality(quality_measure = self.quality_measure, z=self.z, 
                                 comparison_type = self.comparison_type , size_corr = self.size_corr)
            res = bus_rand.quals
            results = results+res
        
            self.random_output = results
            
        pvalues = []
        for x in self.quals:
            empirical_p_value = (sum([(comp > x) for comp in self.random_output])+0.5*sum([(comp == x) for comp in self.random_output])+1)/(len(self.random_output)+1)
            pvalues.append(empirical_p_value)
        self.pvalues = pvalues
        
        print('after checking ',self.quality_measure_counter,' potential subgroups')
        print('Outcome of bottumUpSearch is:')
        print(' ')
        print('avg_quality = ',round(self.avg_quality,3))
        print('max_quality = ',round(max(self.quals),3))
        print(' ')
        print('avg_coverage = ',round(self.avg_coverage,3))
        print('max_coverage = ',round(max(self.covs),3))
        print(' ')
            
        p=0
        for z in self.result:
            
            conjunction = " Ʌ ".join(["(" + condition.replace(" == ", "=").strip() + ")" for condition in z[1]])
            print('description =',conjunction)
            print('quality =',round(z[0],3)," with p-value= ",round(self.pvalues[p],4))
            print('coverage =',round(z[2],3))
            print(' ')
            p+=1
        to_be_saved = [0,0]
        to_be_saved[0] = self.pvalues
        to_be_saved[1] = self.random_output

        np.save(save_title, to_be_saved)
        
    
    def save(self, title, **kwargs):
        
        self.title = title
            
        try:
            pickle_results = pd.read_pickle('results_hierarchical.pkl')
        except:
            pickle_results = pd.DataFrame(columns=['title', 'running_time', "subgroups_checked", 'descs', 'avg_quality', 'avg_coverage',
                                                   'quality_measure', 'x', 'd', 'q',
                                                   'correct_for_size_var', 'result'])

        pickle_results.loc[len(pickle_results)] = [self.title, self.running_time, self.quality_measure_counter, self.descs, self.avg_quality, self.avg_coverage,
                                                   self.quality_measure, self.x, self.d, self.q,
                                                   self.size_corr, self.result]

        pickle_results.to_pickle('results_hierarchical.pkl')
        
        try:
            try:
                pickle_results = pd.read_pickle('random_results_hierarchical.pkl')
            except:
                pickle_results = pd.DataFrame(columns=['title', 'p-values','results'])

            pickle_results.loc[len(pickle_results)] = [self.title, self.pvalues, self.random_output]

            pickle_results.to_pickle('random_results_hierarchical.pkl')
        except:
            pass

        

In [4]:
df = pd.read_pickle("dutch_voting_db.pkl")
df = df.drop(columns=['OngeldigeStemmen','BlancoStemmen','GeldigeStemmen','Region code','Neighbourhoods (nr.)','Kiesgerechtigden','Opkomst'])
df = df.rename(columns=lambda x: re.sub(r'\W+', '', x))
df = df.rename(columns=lambda x: re.sub(r'^(\d+)(.*)$', r'\2\1', x))
df = df.reset_index(drop=True)

In [5]:
# Convert the DataFrame columns into a single 'target' column with arrays
targets = df.columns[-37:]
features = df.columns[:-37]

targets_l = list(targets)
df['target'] = df[targets_l].values.tolist()

# Drop the original columns
df = df.drop(columns=targets_l)
matrix = calculate_distance_matrix(df)

# BUS Coverage 0.01 , z=3

Table 6.13 - third column

In [6]:
df=df.reset_index(drop=True)
bus = botumUpSearch(df, matrix, 20, 3, 20, nr_chunks=10, min_coverage_perc=0.01, min_coverage_abs=3)
bus.findQuality(quality_measure = cluster_based_quality_measure,z=3 ,comparison_type = "complement" , size_corr = no_size_corr)

promising_combinations_start step=  20
promising_combinations_mid step=  40
promising_combinations_large=  25
descriptions 75
unique descriptions 1184


In [7]:
bus.quals

[1.293951196573084,
 1.314950475672752,
 1.298134364754195,
 1.3167576771831604,
 1.3552098316103516,
 1.3035212166491437,
 1.3487951949724246,
 1.332636936330153,
 1.3174888793232775,
 1.3678380096870426,
 1.4533540569276246,
 1.5504283428087746,
 1.3240684591318628,
 1.3537530307708934,
 1.427718742456057,
 1.5826189239583872,
 1.3889600052197488,
 1.331594824491348,
 1.3533120237854914,
 1.6915378397931546]

In [9]:
bus.print_outcome()

after checking  1145  potential subgroups, in  24.26  seconds
Outcome of bottumUpSearch is:
 
avg_quality =  1.385
max_quality =  1.692
 
avg_coverage =  0.013
max_coverage =  0.017
 
description = (321.0 <= Averagehousepricex1000EUR <= 351.0) Ʌ (0.107289107 <= Companiesbytypefinancialservicesandrealestate <= 0.113445378) Ʌ (0.0 <= Ducksforslaughternr <= 0.0)
quality = 1.692
coverage = 0.011
 
description = (0.136296732 <= Studentsbol <= 0.158375635) Ʌ (0.167312161 <= Companiesbytypetradeandcateringindustry <= 0.182894029) Ʌ (0.3 <= Cultivatedlandbytypehorticultureopenground <= 0.7)
quality = 1.583
coverage = 0.011
 
description = (321.0 <= Averagehousepricex1000EUR <= 351.0) Ʌ (0.0 <= Furanimalsnr <= 0.0) Ʌ (4.4 <= Migrationbackgroundremainingnonwestern <= 5.1)
quality = 1.55
coverage = 0.014
 
description = (7.0 <= Pigsnr <= 1020.0) Ʌ (0.327256153 <= Companiesbytypebusinessservices <= 0.442088091) Ʌ (0.0 <= Ducksforslaughternr <= 0.0)
quality = 1.453
coverage = 0.011
 
description = 

# BUS Coverage 0.01 , z=2

Table 6.13 - second column

In [10]:
st=time.time()
bus = botumUpSearch(df, matrix, 20, 3, 20, nr_chunks=10, min_coverage_perc=0.01, min_coverage_abs=3)
bus.findQuality(quality_measure = cluster_based_quality_measure,z=2 ,comparison_type = "complement" , size_corr = no_size_corr)
print('tijd= ',time.time()-st)

promising_combinations_start step=  20
promising_combinations_mid step=  20
promising_combinations_large=  20
descriptions 60
unique descriptions 21928
tijd=  162.53642225265503


In [11]:
bus.quals

[1.517455358640985,
 1.5477984455292113,
 1.5377042307187123,
 1.5552446009759537,
 1.5504283428087746,
 1.5476323109398813,
 1.5808341069512577,
 1.5826189239583872,
 1.5584509946203557,
 1.5757709799281197,
 1.6584031942036483,
 1.5599396905223128,
 1.5570511137900664,
 1.6915378397931546,
 1.6368470211448956,
 1.6406266846444524,
 1.5881834228211051,
 1.6903988589593832,
 1.5631361676297708,
 1.57999988448734]

In [12]:
bus.print_outcome()


after checking  14696  potential subgroups, in  161.941  seconds
Outcome of bottumUpSearch is:
 
avg_quality =  1.586
max_quality =  1.692
 
avg_coverage =  0.012
max_coverage =  0.014
 
description = (321.0 <= Averagehousepricex1000EUR <= 351.0) Ʌ (0.107289107 <= Companiesbytypefinancialservicesandrealestate <= 0.113445378) Ʌ (0.0 <= Ducksforslaughternr <= 0.0)
quality = 1.692
coverage = 0.011
 
description = (321.0 <= Averagehousepricex1000EUR <= 351.0) Ʌ (0.051498847 <= Studentsbbl <= 0.058400561) Ʌ (0.0 <= Ducksforslaughternr <= 0.0)
quality = 1.69
coverage = 0.011
 
description = (0.087837838 <= Companiesbytypetransportinformationandcomunication <= 0.102040816) Ʌ (35.0 <= Motorcyclesper1000inhabitants <= 40.0) Ʌ (4.4 <= Migrationbackgroundremainingnonwestern <= 5.1)
quality = 1.658
coverage = 0.011
 
description = (29.3 <= Householdswithoutchildren <= 30.5) Ʌ (0.107289107 <= Companiesbytypefinancialservicesandrealestate <= 0.113445378) Ʌ (0.0 <= Ducksforslaughternr <= 0.0)
quality

# BS Coverage 0.01 

Table 6.13 - first column

In [13]:
st=time.time()
outputEMM = EMM(bus.df, features, w=30, d=3, q=20, quality_measure=cluster_based_quality_measure, catch_all_description=[],                
        comparison_type = 'complement',target='target', n_chunks=5, ensure_diversity=True,
        report_progress=False, allow_exclusion=False, min_coverage = 0.01,
                          min_coverage_abs = 3, min_error = 0.01,distance_matrix=matrix,correct_for_size=no_size_corr)
print('tijd= ',time.time()-st)
analyseEMMresults(outputEMM)

level :  0
    seed :  []
0 / 3 queue= 1 / 1
eta  []
level :  1
    seed :  ["Companiesbytypeagricultureforestryandfishery == '0.006140022 <= Companiesbytypeagricultureforestryandfishery <= 0.014687882'"]
1 / 3 queue= 1 / 30
eta  ["Companiesbytypeagricultureforestryandfishery == '0.006140022 <= Companiesbytypeagricultureforestryandfishery <= 0.014687882'"]
    seed :  ["Men == '0.470656028 <= Men <= 0.488590802'"]
1 / 3 queue= 2 / 30
eta  ["Men == '0.470656028 <= Men <= 0.488590802'"]
    seed :  ["Companiesbytypetransportinformationandcomunication == '0.102040816 <= Companiesbytypetransportinformationandcomunication <= 0.155615697'"]
1 / 3 queue= 3 / 30
eta  ["Companiesbytypetransportinformationandcomunication == '0.102040816 <= Companiesbytypetransportinformationandcomunication <= 0.155615697'"]
    seed :  ["Motorcyclesper1000inhabitants == '32.0 <= Motorcyclesper1000inhabitants <= 35.0'"]
1 / 3 queue= 4 / 30
eta  ["Motorcyclesper1000inhabitants == '32.0 <= Motorcyclesper1000inhabit

    seed :  ["MigrationbackgroundMaroccan == '1.1 <= MigrationbackgroundMaroccan <= 1.9'", "Householdswithchildren == '36.1 <= Householdswithchildren <= 37.1'"]
2 / 3 queue= 7 / 30
eta  ["MigrationbackgroundMaroccan == '1.1 <= MigrationbackgroundMaroccan <= 1.9'", "Householdswithchildren == '36.1 <= Householdswithchildren <= 37.1'"]
    seed :  ["Motorcyclesper1000inhabitants == '32.0 <= Motorcyclesper1000inhabitants <= 35.0'", "Goatsnr == '5.0 <= Goatsnr <= 13.0'"]
2 / 3 queue= 8 / 30
eta  ["Motorcyclesper1000inhabitants == '32.0 <= Motorcyclesper1000inhabitants <= 35.0'", "Goatsnr == '5.0 <= Goatsnr <= 13.0'"]
    seed :  ["MigrationbackgroundformerDutchAntillesAruba == '1.2 <= MigrationbackgroundformerDutchAntillesAruba <= 4.1'", "Totalroadlengthkm == '227.0 <= Totalroadlengthkm <= 261.0'"]
2 / 3 queue= 9 / 30
eta  ["MigrationbackgroundformerDutchAntillesAruba == '1.2 <= MigrationbackgroundformerDutchAntillesAruba <= 4.1'", "Totalroadlengthkm == '227.0 <= Totalroadlengthkm <= 261.0'

We checked  10327  subgroups
tijd=  172.30108857154846
Outcome of EMM is:
 
avg_quality =  1.536
max_quality =  1.637
 
avg_coverage =  0.012
max_coverage =  0.014
 
description = (Companiesbytypebusinessservices='0.327256153 <= Companiesbytypebusinessservices <= 0.442088091') Ʌ (Companiesbytypeagricultureforestryandfishery='0.023679417 <= Companiesbytypeagricultureforestryandfishery <= 0.037234043') Ʌ (Ducksforslaughternr='0.0 <= Ducksforslaughternr <= 0.0')
quality = 1.637
coverage = 0.011
 
description = (MigrationbackgroundMaroccan='1.1 <= MigrationbackgroundMaroccan <= 1.9') Ʌ (Averagehousepricex1000EUR='321.0 <= Averagehousepricex1000EUR <= 351.0')
quality = 1.588
coverage = 0.011
 
description = (Companiesbytypebusinessservices='0.327256153 <= Companiesbytypebusinessservices <= 0.442088091') Ʌ (Chickensnr='45596.0 <= Chickensnr <= 119738.0')
quality = 1.581
coverage = 0.011
 
description = (Companiesbytypeagricultureforestryandfishery='0.006140022 <= Companiesbytypeagriculturefo

# BUS Coverage 0.1 , z=3

Table 6.13 - sixth column

In [14]:
st=time.time()
bus = botumUpSearch(df, matrix, 20, 3, 20, nr_chunks=10, min_coverage_perc=0.1, min_coverage_abs=3)
bus.findQuality(quality_measure = cluster_based_quality_measure,z=3 ,comparison_type = "complement" , size_corr = no_size_corr)
print('tijd= ',time.time()-st)

promising_combinations_start step=  20
promising_combinations_mid step=  40
promising_combinations_large=  25
descriptions 75
unique descriptions 1184
tijd=  21.882018327713013


In [15]:
bus.quals

[1.0326152553493673,
 1.0341286131465837,
 1.0330377365824344,
 1.0368466680792554,
 1.0380637495043656,
 1.0331342668075132,
 1.0380791461660102,
 1.0394549863543918,
 1.0395736847745907,
 1.0407800526419306,
 1.0530423365231607,
 1.0394549863543918,
 1.0397272596540477,
 1.0452626853411962,
 1.100045488255923,
 1.0439082686446859,
 1.0523045350361628,
 1.1015818861719153,
 1.053133600626478,
 1.0505740218769735]

In [16]:
bus.print_outcome()

after checking  165  potential subgroups, in  21.314  seconds
Outcome of bottumUpSearch is:
 
avg_quality =  1.047
max_quality =  1.102
 
avg_coverage =  0.218
max_coverage =  0.39
 
description = (0.0 <= Otherpoultrynr <= 0.0) Ʌ (1.1 <= MigrationbackgroundMaroccan <= 1.9)
quality = 1.102
coverage = 0.105
 
description = (1.1 <= MigrationbackgroundMaroccan <= 1.9) Ʌ (0.0 <= Ducksforslaughternr <= 0.0)
quality = 1.1
coverage = 0.103
 
description = (0.0 <= Pigsnr <= 0.0) Ʌ (0.0 <= Turkeysnr <= 0.0) Ʌ (0.0 <= Furanimalsnr <= 0.0)
quality = 1.053
coverage = 0.208
 
description = (0.0 <= Rabbitsnr <= 0.0) Ʌ (5.7 <= Widowed <= 6.0) Ʌ (0.0 <= Furanimalsnr <= 0.0)
quality = 1.053
coverage = 0.105
 
description = (0.0 <= Chickensnr <= 0.0) Ʌ (0.0 <= Pigsnr <= 0.0) Ʌ (0.0 <= Ducksforslaughternr <= 0.0)
quality = 1.052
coverage = 0.185
 
description = (0.0 <= Pigsnr <= 0.0) Ʌ (0.0 <= Furanimalsnr <= 0.0) Ʌ (0.0 <= Ducksforslaughternr <= 0.0)
quality = 1.051
coverage = 0.205
 
description = (0.0 

# BUS Coverage 0.1 , z=2

Table 6.13 - fifth column

In [17]:
st=time.time()
bus = botumUpSearch(df, matrix, 20, 3, 20, nr_chunks=10, min_coverage_perc=0.1, min_coverage_abs=3)
bus.findQuality(quality_measure = cluster_based_quality_measure,z=2 ,comparison_type = "complement" , size_corr = no_size_corr)
print('tijd= ',time.time()-st)

promising_combinations_start step=  20
promising_combinations_mid step=  20
promising_combinations_large=  20
descriptions 60
unique descriptions 21928
tijd=  139.64768981933594


In [18]:
bus.quals

[1.0530423365231607,
 1.0547261730085356,
 1.053133600626478,
 1.0565715301504337,
 1.079333050874663,
 1.082569063594172,
 1.0667944994287015,
 1.077911327813045,
 1.073605846685641,
 1.0901571089596018,
 1.1015818861719153,
 1.0942024764975893,
 1.1088432372328534,
 1.0897743504698654,
 1.0724407036276233,
 1.0823906216659582,
 1.0810570718782997,
 1.088479768957727,
 1.0753699191359167,
 1.100045488255923]

In [19]:
bus.print_outcome()

after checking  542  potential subgroups, in  139.075  seconds
Outcome of bottumUpSearch is:
 
avg_quality =  1.079
max_quality =  1.109
 
avg_coverage =  0.111
max_coverage =  0.208
 
description = (0.0 <= Otherpoultrynr <= 0.0) Ʌ (77.6 <= Dutchbackground <= 81.4)
quality = 1.109
coverage = 0.103
 
description = (0.0 <= Otherpoultrynr <= 0.0) Ʌ (1.1 <= MigrationbackgroundMaroccan <= 1.9)
quality = 1.102
coverage = 0.105
 
description = (0.0 <= Otherpoultrynr <= 0.0) Ʌ (1.1 <= MigrationbackgroundMaroccan <= 1.9) Ʌ (0.0 <= Ducksforslaughternr <= 0.0)
quality = 1.1
coverage = 0.103
 
description = (0.0 <= Otherpoultrynr <= 0.0) Ʌ (0.0 <= Rabbitsnr <= 0.0) Ʌ (1.1 <= MigrationbackgroundMaroccan <= 1.9)
quality = 1.094
coverage = 0.103
 
description = (411.0 <= Privatelyownedcarsper1000inhabitants <= 442.0) Ʌ (0.0 <= Turkeysnr <= 0.0) Ʌ (0.0 <= Furanimalsnr <= 0.0)
quality = 1.09
coverage = 0.103
 
description = (0.5 <= MigrationbackgroundformerDutchAntillesAruba <= 0.7) Ʌ (0.0 <= Turkeysnr

# BS Coverage 0.1 

Table 6.13 - fourth column

In [20]:
st=time.time()
outputEMM = EMM(bus.df, features, w=30, d=3, q=20, quality_measure=cluster_based_quality_measure, catch_all_description=[],                
        comparison_type = 'complement',target='target', n_chunks=5, ensure_diversity=True,
        report_progress=False, allow_exclusion=False, min_coverage = 0.1,
                          min_coverage_abs = 3, min_error = 0.01,distance_matrix=matrix,correct_for_size=no_size_corr)
print('tijd= ',time.time()-st)
analyseEMMresults(outputEMM)

level :  0
    seed :  []
0 / 3 queue= 1 / 1
eta  []
level :  1
    seed :  ["Cultivatedlandbytypegreenfoddercrops == '0.7 <= Cultivatedlandbytypegreenfoddercrops <= 3.9'"]
1 / 3 queue= 1 / 30
eta  ["Cultivatedlandbytypegreenfoddercrops == '0.7 <= Cultivatedlandbytypegreenfoddercrops <= 3.9'"]
    seed :  ["yearsold4564 == '26.0 <= yearsold4564 <= 27.6'"]
1 / 3 queue= 2 / 30
eta  ["yearsold4564 == '26.0 <= yearsold4564 <= 27.6'"]
    seed :  ["Motorcyclesper1000inhabitants == '35.0 <= Motorcyclesper1000inhabitants <= 40.0'"]
1 / 3 queue= 3 / 30
eta  ["Motorcyclesper1000inhabitants == '35.0 <= Motorcyclesper1000inhabitants <= 40.0'"]
    seed :  ["Studentsbbl == '0.004115226 <= Studentsbbl <= 0.040009729'"]
1 / 3 queue= 4 / 30
eta  ["Studentsbbl == '0.004115226 <= Studentsbbl <= 0.040009729'"]
    seed :  ["Cultivatedlandbytypetemporarygrassland == '0.0 <= Cultivatedlandbytypetemporarygrassland <= 1.2'"]
1 / 3 queue= 5 / 30
eta  ["Cultivatedlandbytypetemporarygrassland == '0.0 <= Cultiv

    seed :  ["Householdswithoutchildren == '20.2 <= Householdswithoutchildren <= 27.4'", "Furanimalsnr == '0.0 <= Furanimalsnr <= 0.0'"]
2 / 3 queue= 12 / 13
eta  ["Householdswithoutchildren == '20.2 <= Householdswithoutchildren <= 27.4'", "Furanimalsnr == '0.0 <= Furanimalsnr <= 0.0'"]
    seed :  ["MigrationbackgroundformerDutchAntillesAruba == '0.5 <= MigrationbackgroundformerDutchAntillesAruba <= 0.7'", "Turkeysnr == '0.0 <= Turkeysnr <= 0.0'"]
2 / 3 queue= 13 / 13
eta  ["MigrationbackgroundformerDutchAntillesAruba == '0.5 <= MigrationbackgroundformerDutchAntillesAruba <= 0.7'", "Turkeysnr == '0.0 <= Turkeysnr <= 0.0'"]
We checked  405  subgroups
tijd=  180.70232391357422
Outcome of EMM is:
 
avg_quality =  1.071
max_quality =  1.109
 
avg_coverage =  0.114
max_coverage =  0.208
 
description = (Dutchbackground='77.6 <= Dutchbackground <= 81.4')
quality = 1.109
coverage = 0.103
 
description = (MigrationbackgroundMaroccan='1.1 <= MigrationbackgroundMaroccan <= 1.9')
quality = 1.102